# Snowflake Python: reuse your existing CLI TOML

Yes, reuse the file you already created for Snowflake CLI. This notebook reads it without copying credentials or changing the file. Run locally in Jupyter/VS Code on the machine where your TOML is stored.

| Existing file | Connection section | How Python uses it |
|---|---|---|
| `~/.config/snowflake/config.toml` | `[connections.classroom]` | Read the selected table and pass its settings to the connector |
| `~/.config/snowflake/connections.toml` | `[classroom]` | The connector supports this format natively; the loader below also handles it |

`~/.config/snowflake/` is a directory; specify the actual TOML filename below. Use the path reported by your CLI setup if different. This notebook explicitly selects a path, avoiding differences in automatic discovery across operating systems.

[Snowflake CLI connection formats](https://docs.snowflake.com/en/developer-guide/snowflake-cli/connecting/configure-connections) ? [Python connector connections](https://docs.snowflake.com/en/developer-guide/python-connector/python-connector-connect)

## 1. Use your notebook kernel

The CLI installation may be in a different Python environment. Install into the current kernel with `%pip`. If you already have these packages, skip this cell. Restart the kernel if needed. `tomli` supplies TOML parsing on Python versions before 3.11.

In [ ]:
%pip install snowflake-connector-python tomli

In [ ]:
import os
import sys
from pathlib import Path
from getpass import getpass
import snowflake.connector
try:
    import tomllib
except ModuleNotFoundError:
    import tomli as tomllib

print("Python:", sys.executable)
print("Connector:", snowflake.connector.__version__)

## 2. Select the existing file and connection

Change only the path and connection name if your CLI setup used different ones. The companion CLI notebook used `classroom`. On Windows, you can supply an absolute path such as `Path(r"C:\Users\YOUR_USER\.config\snowflake\config.toml")`.

If both files exist, the CLI uses connections from `connections.toml`; select that file here to use the same definitions. A Linux/WSL home directory and a Windows home directory are different: run the kernel on the same system as the CLI or supply the accessible absolute path.

In [ ]:
TOML_PATH = Path("~/.config/snowflake/config.toml").expanduser()
CONNECTION_NAME = "classroom"

# For the shared format, change the filename above to connections.toml.
if not TOML_PATH.is_file():
    raise FileNotFoundError(
        f"TOML file not found at {TOML_PATH}. Set TOML_PATH to your existing CLI file."
    )

In [ ]:
def load_connection(path, name):
    with path.open("rb") as stream:
        settings = tomllib.load(stream)
    # CLI config.toml nests profiles under connections; shared files do not.
    profiles = settings.get("connections", settings)
    if not isinstance(profiles, dict):
        raise ValueError("Expected a TOML table containing connection profiles.")
    if name not in profiles or not isinstance(profiles[name], dict):
        available = ", ".join(k for k, v in profiles.items() if isinstance(v, dict))
        raise ValueError(f"Connection {name!r} not found. Available tables: {available}")
    return dict(profiles[name])

connection_params = load_connection(TOML_PATH, CONNECTION_NAME)
print("Loaded connection:", CONNECTION_NAME)
# Do not display connection_params: it can contain credentials.

## 3. Reuse authentication and connect

The loader preserves your profile's account, user, role, warehouse, database, schema and authentication settings. It does not implement every CLI environment override. It explicitly supports the password variable from the companion exercise, `SNOWFLAKE_CONNECTIONS_CLASSROOM_PASSWORD`, with priority over a password in TOML.

An exported variable reaches Jupyter only if Jupyter was launched from that terminal. If missing, password authentication prompts privately. Browser SSO and key-pair profiles retain their configured authentication. This opens a new session; an earlier CLI login does not automatically authenticate Python.

For password authentication, optionally supply a fresh MFA code when prompted; otherwise complete the configured challenge. Keep the authentication method required by your account. If your CLI workflow depends on additional environment overrides or CLI-only options, supply the corresponding connector settings explicitly.

In [ ]:
password_env = f"SNOWFLAKE_CONNECTIONS_{CONNECTION_NAME.upper()}_PASSWORD"
if os.environ.get(password_env):
    connection_params["password"] = os.environ[password_env]

# Expand home-directory notation for a key file, if the profile uses one.
if connection_params.get("private_key_file"):
    connection_params["private_key_file"] = str(
        Path(connection_params["private_key_file"]).expanduser()
    )

authenticator = str(connection_params.get("authenticator", "snowflake")).lower()
uses_key = bool(connection_params.get("private_key_file") or connection_params.get("private_key"))
try:
    if authenticator in {"snowflake", "username_password_mfa"} and not uses_key:
        if not connection_params.get("password"):
            connection_params["password"] = getpass("Snowflake password: ")
        if not connection_params.get("passcode"):
            connection_params["passcode"] = getpass(
                "MFA code, or Enter for your configured authentication flow: "
            ) or None
    conn = snowflake.connector.connect(**connection_params)
finally:
    connection_params.clear()

print("Connected.")

## 4. Verify context and run a query

These queries need no pre-existing orders table. The role and context come from your TOML profile, not from a Snowsight worksheet. The connection closes even if a query fails. Re-run sections 2?3 before executing this cell again.

In [ ]:
try:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT CURRENT_USER(), CURRENT_ROLE(), CURRENT_WAREHOUSE(),
                   CURRENT_DATABASE(), CURRENT_SCHEMA(), CURRENT_VERSION()
        """)
        print("Context:", cur.fetchone())

        cur.execute("""
            SELECT COLUMN1::NUMBER AS ORDER_ID,
                   COLUMN2::VARCHAR AS STATUS,
                   COLUMN3::NUMBER(12,2) AS ORDER_TOTAL
            FROM VALUES (1001, 'NEW', 120.50),
                        (1002, 'SHIPPED', 250.00),
                        (1003, 'NEW', 75.25)
            WHERE COLUMN2 = %s
            ORDER BY ORDER_ID
        """, ("NEW",))
        print([column[0] for column in cur.description])
        for row in cur.fetchall():
            print(row)
        print("Query ID:", cur.sfqid)
finally:
    conn.close()
    print("Connection closed.")

Expected orders: **1001** and **1003**. After completing the S3 exercise, you can replace the second query with:

```sql
SELECT ORDER_ID, STATUS, ORDER_TOTAL
FROM ORDERS
WHERE STATUS = %s
ORDER BY ORDER_ID
LIMIT 100
```

Keep the bound parameter tuple `("NEW",)` and use the profile's existing database/schema/warehouse.

## Optional: native shared-file connection

If your file is already **connections.toml** with a `[classroom]` section, Python can load it directly:

```python
with snowflake.connector.connect(
    connection_name="classroom",
    connections_file_path=str(Path("~/.config/snowflake/connections.toml").expanduser()),
) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT CURRENT_USER()")
        print(cur.fetchone())
```

This short example assumes the profile provides the required authentication settings. Pass `password=getpass("Snowflake password: ")` explicitly if needed; the connector does not automatically apply the CLI's named password environment variable. Do not use CLI-style `[connections.classroom]` sections with this native API.

No conversion is necessary for this notebook: sections 2?3 already read either format.

## Troubleshooting

| Symptom | Check |
|---|---|
| File not found | Exact filename, kernel machine, and `Path.home()`; `~` refers to the kernel user's home |
| Profile not found | Set `CONNECTION_NAME` to your existing profile name |
| CLI works, Python asks for password | Launch Jupyter from the exported-variable terminal or use the hidden prompt |
| Authentication fails | Check the profile's authentication method and complete MFA; other CLI environment overrides are not applied by this loader |
| No warehouse / object not found | Check the profile's warehouse, database, schema and role |
| Key file not found | Use an accessible absolute key-file path in your profile |

Only local syntax and configuration-loader checks were run during preparation. Live authentication and SQL execution require your account. Keep TOML credentials outside the course repository and avoid displaying them in notebook outputs.